# Análisis de Performance Comercial — Olist Marketplace (2016–2018)

**Autor:** Ana Laura Marín Sánchez 
**Fecha:** Abril 2026 
**Dataset:** Olist Brazilian E-Commerce — [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## Resumen Ejecutivo

[Completar con los hallazgos principales del análisis al finalizar el reporte]

---

## Contexto del Negocio

Olist es una startup brasileña fundada en 2015 que opera como intermediario entre pequeños comercios y los grandes marketplaces de e-commerce del país (Mercado Libre, B2W, Via Varejo, entre otros). El modelo permite a vendedores individuales publicar su catálogo en múltiples plataformas mediante un único contrato, con logística y servicio al cliente centralizados.

El dataset cubre el período septiembre 2016 – octubre 2018 e incluye datos reales de transacciones, ciclos de entrega y reseñas de clientes. La base de datos comprende:

- **pedidos**: ~99,000 órdenes con timestamps del ciclo de vida completo
- **clientes**: ~99,000 compradores distribuidos por estados brasileños
- **items_pedido**: ~113,000 líneas de producto con precio y flete por vendedor
- **pagos**: ~104,000 registros con método de pago, cuotas y monto
- **resenas**: ~98,000 evaluaciones (puntuación 1–5) con timestamps de respuesta
- **productos**: ~33,000 SKUs con categoría, dimensiones y peso
- **vendedores**: ~3,100 vendedores activos en múltiples estados

---

## Configuración del Entorno

Conexión a la base de datos y verificación del esquema disponible antes de iniciar el análisis.

In [98]:
# Importar librerias necesarias
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuracion de visualizacion
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Conexion a la base de datos
DB_PATH = "data/portafolio_olist.db"

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"No se encontro la base de datos en '{DB_PATH}'. "
        "Ejecutar primero: python data/lab1-generacion-datasets.py"
    )

conn = sqlite3.connect(DB_PATH)


def q(query):
    """Ejecuta una consulta SQL y retorna el resultado como DataFrame."""
    return pd.read_sql_query(query, conn)


# Verificar tablas disponibles
print("Tablas disponibles:")
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").to_string(index=False))


Tablas disponibles:
        name
    clientes
items_pedido
       pagos
     pedidos
   productos
     resenas
  vendedores


---

## 1. Caracterización del Período de Análisis

### 1.1 Resumen Operacional

Antes de cualquier segmentación, un reporte ejecutivo requiere establecer la escala de la operación. La tabla pedidos registra cada orden procesada, pagos consolida el valor monetario de cada transacción en el campo valor_pago, clientes almacena la identidad única del comprador en cliente_unico_id, e items_pedido vincula cada orden con el vendedor que la atendió. Cruzando estas cuatro tablas se obtienen los denominadores de todo el análisis: volumen total, revenue bruto, cobertura de clientes y tamaño activo de la red de vendedores.


In [99]:
print(q("""
WITH pagos_por_pedido AS (
    SELECT
        pedido_id,
        SUM(valor_pago) AS valor_pago_total
    FROM pagos
    GROUP BY pedido_id
)
SELECT
    COUNT(DISTINCT p.pedido_id) AS volumen_total,
    ROUND(SUM(pp.valor_pago_total), 2) AS revenue_bruto,
    COUNT(DISTINCT c.cliente_unico_id) AS cobertura_clientes,
    COUNT(DISTINCT i.vendedor_id) AS vendedores_activos
FROM pedidos p
INNER JOIN pagos_por_pedido pp
    ON p.pedido_id = pp.pedido_id
INNER JOIN clientes c
    ON p.cliente_id = c.cliente_id
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id;
""").to_string(index=False))

 volumen_total  revenue_bruto  cobertura_clientes  vendedores_activos
         98665    20308134.71               95419                3095


### 1.2 Evolución Mensual de Pedidos y Revenue

El e-commerce brasileño muestra estacionalidad marcada por el Black Friday en noviembre y las fiestas de fin de año. La tabla pedidos registra la fecha de cada orden en fecha_compra, y la tabla pagos acumula el valor de cada transacción en valor_pago. Agrupando ambas por mes se construye la serie temporal que permite identificar picos de demanda, meses de contracción y la tendencia de crecimiento general durante el período 2016–2018.


In [100]:
print(q("""
SELECT
    strftime('%Y-%m', p.fecha_compra) AS mes,
    COUNT(*) AS num_pedidos,
    SUM(pa.valor_pago) AS revenue
FROM pedidos p
INNER JOIN pagos pa
    ON p.pedido_id = pa.pedido_id
WHERE p.fecha_compra BETWEEN '2016-01-01' AND '2018-12-31'
GROUP BY mes
ORDER BY mes ASC;
""").to_string(index=False))

    mes  num_pedidos    revenue
2016-09            3     252.24
2016-10          342   59090.48
2016-12            1      19.62
2017-01          850  138488.04
2017-02         1886  291908.01
2017-03         2837  449863.60
2017-04         2571  417788.03
2017-05         3944  592918.82
2017-06         3436  511276.38
2017-07         4317  592382.92
2017-08         4550  674396.32
2017-09         4516  727762.45
2017-10         4860  779677.88
2017-11         7863 1194882.80
2017-12         5895  878401.48
2018-01         7563 1115004.18
2018-02         6952  992463.34
2018-03         7512 1159652.12
2018-04         7209 1160785.48
2018-05         7135 1153982.15
2018-06         6419 1023880.50
2018-07         6507 1066540.75
2018-08         6698 1022425.32
2018-09           16    4439.54
2018-10            4     589.67


### 1.3 Distribución por Estado del Pedido

El campo estado en la tabla pedidos registra la fase del ciclo de vida de cada orden: delivered, shipped, canceled, unavailable, invoiced, processing, created y approved. Una distribución operacionalmente sana concentra la mayoría de órdenes en delivered; acumulaciones en estados intermedios señalan cuellos de botella en el proceso. La tasa de cancelación —pedidos en estado canceled sobre el total— es un indicador directo de fricción en la experiencia de compra.


In [102]:
print(q("""
SELECT
    estado,
    COUNT(*) AS num_pedidos,
    ROUND(COUNT() * 100.0 / (SELECT COUNT() FROM pedidos), 2) AS porcentaje_total
FROM pedidos
GROUP BY estado
ORDER BY num_pedidos DESC;
""").to_string(index=False))

     estado  num_pedidos  porcentaje_total
  delivered        96478             97.02
    shipped         1107              1.11
   canceled          625              0.63
unavailable          609              0.61
   invoiced          314              0.32
 processing          301              0.30
    created            5              0.01
   approved            2              0.00


In [103]:
print(q("""
SELECT
    ROUND(
        SUM(CASE WHEN estado = 'canceled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS tasa_cancelacion
FROM pedidos;
""").to_string(index=False))

 tasa_cancelacion
             0.63


---

## 2. Red de Vendedores

### 2.1 Top 15 Vendedores por Revenue

En modelos de marketplace, el revenue tiende a concentrarse en un grupo pequeño de operadores. La tabla items_pedido registra cada línea de venta con su precio y flete por vendedor, y la tabla vendedores almacena el estado (unidad federativa) de cada uno. Cruzando ambas con pedidos —filtrado a estado = delivered— se obtiene el ranking de los 15 mayores aportantes al revenue y la proporción del total que concentran, un indicador clave de dependencia operacional.


In [104]:
print(q("""
SELECT
    i.vendedor_id,
    v.estado AS estado_vendedor,
    ROUND(SUM(i.precio + i.flete), 2) AS revenue_vendedor,
    ROUND(
        100.0 * SUM(i.precio + i.flete) /
        (
            SELECT SUM(i2.precio + i2.flete)
            FROM pedidos p2
            INNER JOIN items_pedido i2
                ON p2.pedido_id = i2.pedido_id
            WHERE p2.estado = 'delivered'
        ),
        2
    ) AS pct_revenue_total
FROM pedidos p
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id
INNER JOIN vendedores v
    ON i.vendedor_id = v.vendedor_id
WHERE p.estado = 'delivered'
GROUP BY i.vendedor_id, v.estado
ORDER BY revenue_vendedor DESC
LIMIT 15;
""").to_string(index=False))

                     vendedor_id estado_vendedor  revenue_vendedor  pct_revenue_total
4869f7a5dfa277a7dca6462dcf3b52b2              SP         247007.06               1.60
7c67e1448b00f6e969d365cea6b010ab              SP         237806.69               1.54
4a3ca9315b744ce9f8e9374361493884              SP         231220.43               1.50
53243585a1d6dc2643021fd1853d8905              BA         230797.02               1.50
fa1c13f2614d7b5c4749cbc52fecda94              SP         200833.50               1.30
da8622b14eb17ae2831f4ac5b9dab84a              SP         184706.78               1.20
7e93a43ef30c4f03f38b393420bc753a              SP         171973.55               1.12
1025f0e2d44d7041d6cf58b6550e0bfa              SP         171924.96               1.11
7a67c85e85bb2ce8582c35f2203ad736              SP         160278.52               1.04
955fee9216a65b617aa5c0531780ce60              SP         156606.48               1.02
6560211a19b47992c3666cc44a7e94c0              SP      

### 2.2 Distribución Geográfica de Vendedores

La ubicación del vendedor determina los tiempos y costos de entrega. La tabla vendedores registra el estado brasileño de cada operador, e items_pedido contiene el precio y flete de cada transacción. Cruzando ambas se cuantifica cómo se distribuye la red de vendedores por región y si la concentración geográfica se corresponde con la concentración de revenue —es decir, si la desigualdad territorial también es una desigualdad comercial.


In [105]:
print(q("""
SELECT
    v.estado AS estado_vendedor,
    COUNT(DISTINCT v.vendedor_id) AS num_vendedores,
    ROUND(SUM(i.precio + i.flete), 2) AS revenue_total,
    ROUND(
        100.0 * SUM(i.precio + i.flete) /
        (SELECT SUM(precio + flete) FROM items_pedido),
        2
    ) AS pct_revenue
FROM vendedores v
INNER JOIN items_pedido i
    ON v.vendedor_id = i.vendedor_id
GROUP BY v.estado
ORDER BY pct_revenue DESC;
""").to_string(index=False))

estado_vendedor  num_vendedores  revenue_total  pct_revenue
             SP            1849    10235883.88        64.61
             PR             349     1458900.73         9.21
             MG             244     1224159.80         7.73
             RJ             171      937814.12         5.92
             SC             190      738973.13         4.66
             RS             129      435802.63         2.75
             BA              19      305262.24         1.93
             DF              30      116243.54         0.73
             PE               9      103886.31         0.66
             GO              40       78964.71         0.50
             ES              23       59860.74         0.38
             MA               1       48550.24         0.31
             CE              13       24600.47         0.16
             MT               4       21702.45         0.14
             PB               6       18584.15         0.12
             RN               5       11

### 2.3 Volumen de Ventas y Satisfacción

La hipótesis es que los vendedores con mayor volumen tienen procesos más maduros y, por tanto, mejor satisfacción. La tabla items_pedido registra cuántos ítems vendió cada operador, y la tabla resenas almacena la puntuación (escala 1–5) que los clientes asignaron a cada pedido en el campo puntuacion. Vinculando ambas vía pedido_id se contrasta volumen y calidad percibida, restringiéndose a vendedores con al menos 30 reseñas para garantizar representatividad estadística.


In [108]:
print(q("""
WITH volumen AS (
    SELECT
        vendedor_id,
        COUNT(*) AS volumen_ventas
    FROM items_pedido
    GROUP BY vendedor_id
),
pedidos_vendedor AS (
    SELECT DISTINCT
        vendedor_id,
        pedido_id
    FROM items_pedido
),
satisfaccion AS (
    SELECT
        pv.vendedor_id,
        ROUND(AVG(r.puntuacion), 2) AS promedio_satisfaccion,
        COUNT(DISTINCT r.resena_id) AS num_resenas
    FROM pedidos_vendedor pv
    INNER JOIN resenas r
        ON pv.pedido_id = r.pedido_id
    GROUP BY pv.vendedor_id
)
SELECT
    v.vendedor_id,
    v.volumen_ventas,
    s.promedio_satisfaccion,
    s.num_resenas
FROM volumen v
INNER JOIN satisfaccion s
    ON v.vendedor_id = s.vendedor_id
WHERE s.num_resenas >= 30
ORDER BY v.volumen_ventas DESC;
""").to_string(index=False))

                     vendedor_id  volumen_ventas  promedio_satisfaccion  num_resenas
6560211a19b47992c3666cc44a7e94c0            2033                   3.94         1837
4a3ca9315b744ce9f8e9374361493884            1987                   3.83         1783
1f50f920176fa81dab994f9023523100            1931                   4.13         1399
cc419e0650a3c5ba77189a1882b7556a            1775                   4.08         1694
da8622b14eb17ae2831f4ac5b9dab84a            1551                   4.18         1297
955fee9216a65b617aa5c0531780ce60            1499                   4.16         1277
1025f0e2d44d7041d6cf58b6550e0bfa            1428                   4.00          901
7c67e1448b00f6e969d365cea6b010ab            1364                   3.49          971
ea8482cd71df3c1969d7b9473ff13abc            1203                   4.00         1138
7a67c85e85bb2ce8582c35f2203ad736            1171                   4.24         1153
4869f7a5dfa277a7dca6462dcf3b52b2            1156                 

---

## 3. Satisfacción del Cliente

### 3.1 Distribución de Puntuaciones

Las reseñas capturan la percepción del cliente sobre todo el proceso, desde la descripción del producto hasta la entrega. La tabla resenas registra la evaluación en el campo puntuacion (escala 1–5) y la fecha en que el vendedor respondió en fecha_respuesta. La distribución de puntuaciones revela el perfil de satisfacción general, y la tasa de respuesta del vendedor —calculada sobre reseñas con fecha_respuesta no nula— es un indicador secundario de la calidad del servicio post-venta.


In [116]:
print(q("""
SELECT
    puntuacion,
    COUNT(*) AS num_resenas,
    ROUND(COUNT() * 100.0 / (SELECT COUNT() FROM resenas), 2) AS porcentaje_total
FROM resenas
GROUP BY puntuacion
ORDER BY puntuacion;
""").to_string(index=False))

print(q("""
SELECT
    ROUND(AVG(
        CASE 
            WHEN fecha_respuesta IS NOT NULL AND fecha_respuesta != '' 
            THEN 1 
            ELSE 0 
        END
    ) * 100, 2) AS tasa_respuesta_pct
FROM resenas;
""").to_string(index=False))

 puntuacion  num_resenas  porcentaje_total
          1        11282             11.46
          2         3114              3.16
          3         8097              8.23
          4        19007             19.31
          5        56910             57.83
 tasa_respuesta_pct
              100.0


In [ ]:
#Verificando que la tasa de respuesta fuera 100 
print(q("""
SELECT
    COUNT(*) AS total,
    COUNT(fecha_respuesta) AS con_respuesta
FROM resenas;
""").to_string(index=False))

 total  con_respuesta
 98410          98410


### 3.2 Satisfacción por Categoría de Producto

La categoría del producto determina la complejidad logística y el riesgo de incidencias. La tabla productos almacena la categoría de cada SKU en el campo categoria, y la tabla resenas registra la puntuacion por pedido. Ambas se vinculan a través de la cadena resenas → pedidos → items_pedido → productos. El análisis compara la puntuación promedio y la tasa de insatisfacción severa (puntuacion ≤ 2) entre categorías con al menos 100 reseñas, identificando los segmentos que concentran los mayores problemas de calidad percibida.


In [118]:
print(q("""
SELECT
    pr.categoria AS categoria_producto,
    COUNT(DISTINCT r.resena_id) AS num_resenas,
    ROUND(AVG(r.puntuacion), 2) AS promedio_satisfaccion,
    ROUND(
        100.0 * SUM(CASE WHEN r.puntuacion <= 2 THEN 1 ELSE 0 END) 
        / COUNT(DISTINCT r.resena_id),
        2
    ) AS tasa_insatisfaccion_pct
FROM resenas r
INNER JOIN pedidos p
    ON r.pedido_id = p.pedido_id
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id
INNER JOIN productos pr
    ON i.producto_id = pr.producto_id
GROUP BY pr.categoria
HAVING COUNT(DISTINCT r.resena_id) >= 100
ORDER BY promedio_satisfaccion ASC;
""").to_string(index=False))

                     categoria_producto  num_resenas  promedio_satisfaccion  tasa_insatisfaccion_pct
                       office_furniture         1258                   3.49                    34.74
                  fashion_male_clothing          110                   3.64                    33.64
                        fixed_telephony          215                   3.68                    31.16
                                  audio          347                   3.83                    22.48
                                    NaN         1441                   3.84                    24.43
              construction_tools_safety          166                   3.84                    23.49
                           home_confort          396                   3.84                    21.72
                         bed_bath_table         9273                   3.90                    22.29
                  furniture_living_room          415                   3.90                

### 3.3 Impacto del Tiempo de Entrega en la Satisfacción

El cumplimiento de la fecha prometida es el principal driver de satisfacción en e-commerce. La tabla pedidos registra dos fechas clave: fecha_estimada_entrega, que es la promesa comunicada al cliente al momento de la compra, y fecha_entrega_cliente, que registra cuándo el pedido llegó efectivamente. La diferencia entre ambas —en días— clasifica cada orden como anticipada, puntual o con retraso, y se contrasta con la puntuacion en resenas para cuantificar el costo en satisfacción de no cumplir el plazo.


In [119]:
print(q("""
SELECT
    CASE
        WHEN julianday(p.fecha_entrega_cliente) < julianday(p.fecha_estimada_entrega) THEN 'anticipada'
        WHEN julianday(p.fecha_entrega_cliente) = julianday(p.fecha_estimada_entrega) THEN 'puntual'
        ELSE 'con_retraso'
    END AS tipo_entrega,
    COUNT(DISTINCT r.resena_id) AS num_resenas,
    ROUND(AVG(r.puntuacion), 2) AS promedio_satisfaccion
FROM pedidos p
INNER JOIN resenas r
    ON p.pedido_id = r.pedido_id
WHERE p.fecha_entrega_cliente IS NOT NULL
  AND p.fecha_estimada_entrega IS NOT NULL
GROUP BY tipo_entrega
ORDER BY promedio_satisfaccion DESC;
""").to_string(index=False))

tipo_entrega  num_resenas  promedio_satisfaccion
  anticipada        87974                   4.30
 con_retraso         7633                   2.57


---

## 4. Eficiencia Logística

### 4.1 Tasa Global de Entregas a Tiempo (OTD Rate)

El On-Time Delivery Rate mide el porcentaje de pedidos entregados en o antes de la fecha prometida al cliente, y es el KPI logístico de referencia del sector. La tabla pedidos registra la fecha de compra en fecha_compra, la entrega real en fecha_entrega_cliente y el plazo comprometido en fecha_estimada_entrega. Comparando las dos últimas sobre el universo de pedidos con estado = delivered se obtiene el OTD Rate global, el tiempo promedio puerta a puerta y los promedios de adelanto y retraso para cada grupo.


In [120]:
print(q("""
SELECT
    COUNT(*) AS total_pedidos_delivered,
    ROUND(
        100.0 * AVG(
            CASE
                WHEN julianday(fecha_entrega_cliente) <= julianday(fecha_estimada_entrega)
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS otd_rate_pct,
    ROUND(AVG(julianday(fecha_entrega_cliente) - julianday(fecha_compra)), 2) AS tiempo_promedio_puerta_puerta_dias,
    ROUND(AVG(
        CASE
            WHEN julianday(fecha_entrega_cliente) < julianday(fecha_estimada_entrega)
            THEN julianday(fecha_estimada_entrega) - julianday(fecha_entrega_cliente)
            ELSE NULL
        END
    ), 2) AS promedio_adelanto_dias,
    ROUND(AVG(
        CASE
            WHEN julianday(fecha_entrega_cliente) > julianday(fecha_estimada_entrega)
            THEN julianday(fecha_entrega_cliente) - julianday(fecha_estimada_entrega)
            ELSE NULL
        END
    ), 2) AS promedio_retraso_dias
FROM pedidos
WHERE estado = 'delivered'
  AND fecha_compra IS NOT NULL
  AND fecha_entrega_cliente IS NOT NULL
  AND fecha_estimada_entrega IS NOT NULL;
""").to_string(index=False))

 total_pedidos_delivered  otd_rate_pct  tiempo_promedio_puerta_puerta_dias  promedio_adelanto_dias  promedio_retraso_dias
                   96470         91.89                               12.56                   13.01                   9.55


### 4.2 Performance Logística por Estado del Vendedor

La distancia geográfica entre vendedor y cliente es el principal determinante estructural del tiempo de entrega. La tabla vendedores registra el estado de origen de cada operador, items_pedido contiene el flete por transacción, y pedidos almacena las fechas del ciclo de vida. Cruzando estas tres fuentes se desglosa la performance logística por región de despacho: tiempo de entrega promedio, OTD Rate y costo de flete promedio para cada unidad federativa.


In [121]:
print(q("""
SELECT
    v.estado AS estado_vendedor,
    COUNT(DISTINCT p.pedido_id) AS num_pedidos,
    ROUND(AVG(julianday(p.fecha_entrega_cliente) - julianday(p.fecha_compra)), 2) AS tiempo_entrega_promedio_dias,
    ROUND(
        100.0 * AVG(
            CASE
                WHEN julianday(p.fecha_entrega_cliente) <= julianday(p.fecha_estimada_entrega)
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS otd_rate_pct,
    ROUND(AVG(i.flete), 2) AS flete_promedio
FROM pedidos p
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id
INNER JOIN vendedores v
    ON i.vendedor_id = v.vendedor_id
WHERE p.estado = 'delivered'
  AND p.fecha_compra IS NOT NULL
  AND p.fecha_entrega_cliente IS NOT NULL
  AND p.fecha_estimada_entrega IS NOT NULL
GROUP BY v.estado
ORDER BY otd_rate_pct ASC;
""").to_string(index=False))

estado_vendedor  num_pedidos  tiempo_entrega_promedio_dias  otd_rate_pct  flete_promedio
             AM            3                         47.84         33.33           27.27
             MA          389                         17.74         76.37           30.03
             PA            8                         13.43         87.50           19.39
             RN           51                         13.02         89.29           23.29
             CE           87                         17.89         91.11           46.74
             SP        68635                         12.28         91.48           18.42
             RJ         4227                         12.02         91.89           19.49
             MS           49                         12.33         92.00           23.98
             DF          808                         12.53         93.32           20.44
             ES          310                         12.88         93.41           32.72
             PR      

### 4.3 Categorías con Mayor Tasa de Retraso

Ciertas categorías presentan sistemáticamente mayor incidencia de retrasos, por características físicas del producto, complejidad de preparación del envío o distribución geográfica de sus vendedores. La tabla productos almacena la categoria de cada SKU, y pedidos registra la fecha de entrega real y la estimada. Cruzando ambas se calcula, por categoría, la tasa de retraso y el promedio de días de demora en las órdenes que no cumplieron el plazo, sobre categorías con al menos 50 pedidos entregados.


In [122]:
print(q("""
SELECT
    pr.categoria AS categoria_producto,
    COUNT(DISTINCT p.pedido_id) AS num_pedidos_entregados,
    ROUND(
        100.0 * AVG(
            CASE
                WHEN julianday(p.fecha_entrega_cliente) > julianday(p.fecha_estimada_entrega)
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS tasa_retraso_pct,
    ROUND(AVG(
        CASE
            WHEN julianday(p.fecha_entrega_cliente) > julianday(p.fecha_estimada_entrega)
            THEN julianday(p.fecha_entrega_cliente) - julianday(p.fecha_estimada_entrega)
            ELSE NULL
        END
    ), 2) AS promedio_dias_demora
FROM pedidos p
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id
INNER JOIN productos pr
    ON i.producto_id = pr.producto_id
WHERE p.estado = 'delivered'
  AND p.fecha_entrega_cliente IS NOT NULL
  AND p.fecha_estimada_entrega IS NOT NULL
GROUP BY pr.categoria
HAVING COUNT(DISTINCT p.pedido_id) >= 50
ORDER BY tasa_retraso_pct DESC;
""").to_string(index=False))

                     categoria_producto  num_pedidos_entregados  tasa_retraso_pct  promedio_dias_demora
                                  audio                     348             12.71                  8.24
                fashion_underwear_beach                     117             12.60                  6.87
                     christmas_supplies                     125             12.00                 10.86
                        books_technical                     256             11.03                  5.35
                           home_confort                     392             10.26                 12.47
              construction_tools_lights                     242              9.97                  4.44
                                   food                     441              9.82                  9.69
                            electronics                    2517              9.75                  7.73
                                    NaN                    1392 

---

## 5. Categorías y Métodos de Pago

### 5.1 Top Categorías por Revenue

La distribución de revenue por categoría determina la arquitectura comercial del catálogo y orienta las inversiones en desarrollo de oferta. La tabla productos clasifica cada SKU en una categoria, e items_pedido registra el precio y flete de cada línea de venta. Cruzando ambas sobre pedidos entregados se obtiene el ranking de las 15 categorías con mayor aporte al revenue total y la participación porcentual de cada una, diferenciando entre categorías de alto valor unitario y categorías masivas de bajo ticket.


In [123]:
print(q("""
SELECT
    pr.categoria AS categoria_producto,
    ROUND(SUM(i.precio + i.flete), 2) AS revenue_total,
    ROUND(
        100.0 * SUM(i.precio + i.flete) /
        (
            SELECT SUM(i2.precio + i2.flete)
            FROM pedidos p2
            INNER JOIN items_pedido i2
                ON p2.pedido_id = i2.pedido_id
            WHERE p2.estado = 'delivered'
        ),
        2
    ) AS pct_revenue_total
FROM pedidos p
INNER JOIN items_pedido i
    ON p.pedido_id = i.pedido_id
INNER JOIN productos pr
    ON i.producto_id = pr.producto_id
WHERE p.estado = 'delivered'
GROUP BY pr.categoria
ORDER BY revenue_total DESC
LIMIT 15;
""").to_string(index=False))

   categoria_producto  revenue_total  pct_revenue_total
        health_beauty     1412089.53               9.16
        watches_gifts     1264333.12               8.20
       bed_bath_table     1225209.26               7.95
       sports_leisure     1118256.91               7.25
computers_accessories     1032723.77               6.70
      furniture_decor      880329.92               5.71
           housewares      758392.25               4.92
           cool_stuff      691680.89               4.49
                 auto      669454.75               4.34
         garden_tools      567145.68               3.68
                 toys      547061.06               3.55
                 baby      466727.65               3.03
            perfumery      443171.63               2.87
            telephony      379202.62               2.46
     office_furniture      335211.36               2.17


### 5.2 Estructura de Costos por Categoría

El ratio flete/precio mide la carga logística sobre el valor de la transacción. En categorías de bajo precio unitario, el costo de envío puede representar una fracción significativa del total, generando fricción en la conversión y reduciendo el margen efectivo del vendedor. La tabla items_pedido registra el precio y el flete de cada línea de venta, y la tabla productos permite agrupar por categoria. El análisis reporta ese ratio para las 20 categorías con mayor volumen de ventas.


In [124]:
print(q("""
SELECT
    pr.categoria AS categoria_producto,
    COUNT(*) AS volumen_ventas,
    ROUND(AVG(i.precio), 2) AS precio_promedio,
    ROUND(AVG(i.flete), 2) AS flete_promedio,
    ROUND(AVG(i.flete * 1.0 / NULLIF(i.precio, 0)), 4) AS ratio_flete_precio
FROM items_pedido i
INNER JOIN productos pr
    ON i.producto_id = pr.producto_id
GROUP BY pr.categoria
ORDER BY volumen_ventas DESC
LIMIT 20;
""").to_string(index=False))

      categoria_producto  volumen_ventas  precio_promedio  flete_promedio  ratio_flete_precio
          bed_bath_table           11115            93.30           18.42              0.2843
           health_beauty            9670           130.16           18.88              0.3021
          sports_leisure            8641           114.34           19.51              0.2939
         furniture_decor            8334            87.56           20.73              0.3467
   computers_accessories            7827           116.51           18.82              0.2944
              housewares            6964            90.79           20.99              0.4027
           watches_gifts            5991           201.14           16.78              0.1706
               telephony            4545            71.21           15.67              0.5059
            garden_tools            4347           111.63           22.77              0.3320
                    auto            4235           139.96   

### 5.3 Métodos de Pago y Uso de Cuotas

El comportamiento de pago en Brasil combina boleto bancário, débito y crédito con parcelamento, práctica cultural arraigada incluso en compras de montos moderados. La tabla pagos registra el tipo_pago utilizado en cada transacción, el valor en valor_pago y el número de cuotas en cuotas. Agrupando por tipo_pago se obtiene la distribución de volumen y valor por método; analizando la distribución de cuotas para tarjeta de crédito se cuantifica el alcance del financiamiento a plazo en la base de compradores.


In [126]:
print(q("""
SELECT
    tipo_pago,
    COUNT(*) AS num_transacciones,
    ROUND(SUM(valor_pago), 2) AS valor_total_pagado,
    ROUND(AVG(valor_pago), 2) AS ticket_promedio
FROM pagos
GROUP BY tipo_pago
ORDER BY valor_total_pagado DESC;
""").to_string(index=False))

  tipo_pago  num_transacciones  valor_total_pagado  ticket_promedio
credit_card              76795         12542084.19           163.32
     boleto              19784          2869361.27           145.03
    voucher               5775           379436.87            65.70
 debit_card               1529           217989.79           142.57
not_defined                  3                0.00             0.00


In [127]:
print(q("""
SELECT
    cuotas,
    COUNT(*) AS num_transacciones,
    ROUND(SUM(valor_pago), 2) AS valor_total_pagado,
    ROUND(AVG(valor_pago), 2) AS ticket_promedio
FROM pagos
WHERE tipo_pago = 'credit_card'
GROUP BY cuotas
ORDER BY cuotas;
""").to_string(index=False))

 cuotas  num_transacciones  valor_total_pagado  ticket_promedio
      0                  2              188.63            94.31
      1              25455          2440445.43            95.87
      2              12413          1579283.03           127.23
      3              10461          1491103.80           142.54
      4               7098          1163907.61           163.98
      5               5239           961174.30           183.47
      6               3920           822611.81           209.85
      7               1626           305157.39           187.67
      8               4268          1313423.34           307.74
      9                644           131015.92           203.44
     10               5328          2211577.34           415.09
     11                 23             2873.44           124.93
     12                133            42783.24           321.68
     13                 16             2407.40           150.46
     14                 15             2

---

## 6. Síntesis y Recomendaciones Estratégicas

Esta sección consolida los hallazgos clave del análisis y formula recomendaciones accionables para las áreas funcionales relevantes. La síntesis debe basarse directamente en los valores cuantitativos obtenidos en las secciones anteriores.

### 6.1 Hallazgos Principales

Completar con los insights más relevantes identificados a lo largo del análisis:

**Escala y Tendencias del Negocio:**  
[Período cubierto, volumen total de pedidos y revenue, tendencias mensuales, meses pico]

**Estructura de la Red de Vendedores:**  
[Concentración de revenue (top N vendedores = X% del total), distribución geográfica, relación volumen-satisfacción]

**Satisfacción del Cliente:**  
[Puntuación promedio global, categorías mejor y peor valoradas, impacto cuantificado del retraso en la puntuación]

**Eficiencia Logística:**  
[OTD Rate global, tiempo promedio de entrega, estados con mejor y peor performance, categorías con mayor tasa de retraso]

**Categorías y Pagos:**  
[Top categorías por revenue, categorías con ratio flete/precio elevado, distribución de métodos de pago]

---

### 6.2 Recomendaciones por Área Funcional

**Operaciones y Logística:**  
[Basado en análisis de OTD Rate, tiempos por estado y categorías problemáticas]

**Gestión de Vendedores (Seller Success):**  
[Basado en concentración de revenue, distribución geográfica y relación volumen-satisfacción]

**Producto y Catálogo:**  
[Basado en análisis de categorías, ticket promedio y ratio flete/precio]

**Experiencia del Cliente:**  
[Basado en distribución de reseñas, impacto del retraso, categorías con alta insatisfacción]

**Finanzas y Procesamiento de Pagos:**  
[Basado en distribución de métodos y uso de cuotas]

---

## Apéndice: Consultas SQL de Referencia

Documentar las principales consultas SQL utilizadas en el análisis para garantizar la trazabilidad y reproducibilidad del reporte. Cada entrada debe incluir el propósito de la consulta y el hallazgo clave obtenido.

In [ ]:
# Cerrar conexion a la base de datos
conn.close()
print("Analisis completado — conexion cerrada")


Analisis completado — conexion cerrada
